In [20]:
import pandas as pd
import glob
import os
import numpy as np
import shutil

In [21]:
template_path = r"EDDTemplate\ESBasic_TRC Format.xlsx"
template_df = pd.read_excel(template_path, sheet_name="ESBasic_TRC")

In [22]:
dir = "data/granger_pfas/2026"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
excel_paths

['data/granger_pfas/2026\\26D0675 FINAL Basic EDD 18 May 26 1301.xlsx']

In [23]:
file_name = '26D0675 FINAL Basic EDD 18 May 26 1301.xlsx'
_26D0675 = pd.read_excel(f'{dir}\\{file_name}')
_26D0675.shape
lab_sgd = "26D0675"

In [24]:
_26D0675['SAMPLEID'].unique()

<StringArray>
[    'Under Drain',        'Cooper 2',        'Cooper 1',  'Groesbeck Pond',
          'MW-43s',          'MW-43d',          'MW-3sr',          'MW-3dr',
          'MW-23r',          'MW-34s',          'MW-34d',          'MW-30s',
          'MW-30d',          'MW-31s',          'MW-31d',        'Leachate',
      'Trip Blank',          'MW-33s',          'MW-33d',          'MW-32s',
          'MW-32d',         'MW-28sr',         'MW-28dr',         'MW-27sr',
         'MW-27dr',         'MW-26sr',        'MW-26dr2',     'Duplicate B',
          'MW-6sr',          'MW-6dr',          'MW-5sr',          'MW-5dr',
          'MW-42s',          'MW-42d',          'MW-41s',          'MW-41d',
          'MW-40s',          'MW-40d',     'Duplicate C', 'Equipment Blank',
     'Field Blank',     'Duplicate A']
Length: 42, dtype: str

In [25]:
_26D0675.columns

Index(['LABID', 'SAMPLEID', 'DATESAMPLED', 'DATERECEIVED', 'DATEEXTRACTED',
       'DATEANALYZED', 'MATRIX', 'METHOD', 'PARAMETER', 'RESULTS', 'UNITS',
       'DETECTIONLIMITS', 'NOTES', 'DILUTION', 'Total Or Dissolved', 'MCL'],
      dtype='str')

In [26]:
_26D0675.NOTES.unique()

<StringArray>
[nan, 'MS07', 'BS03']
Length: 3, dtype: str

In [27]:
edd_to_db_methods = {
    'EPA 200.8 Rev. 5.4': 'E200.8',
    'EPA 200.7 Rev. 4.4': 'E200.7',
    'EPA 8260D': 'SW8260D',
    'Calculation': 'CALC',
    'EPA 350.1 Rev. 2.0': 'E350.1',
    'SM 2540 C-20': 'SM-2540',
    'SM 2320 B-21': 'SM2320',
    'SM 5310 B-14': 'SM5310',
    'EPA 300.0 Rev. 2.1': 'E300.0',
    'SM 4500-Cl D-21': 'SM4500',
    'EPA 410.4 Rev. 2.0': 'E410.4',
}

In [28]:
chemical_name_dict = {
    "Bicarbonate Alkalinity as CaCO3 at pH 4.5": "Alkalinity, bicarbonate, to pH 4.5"
}

In [29]:
analytes_cas_dict = {
    "Arsenic": "7440-38-2",
    "Boron": "7440-42-8",
    "Beryllium": "7440-41-7",
    "Calcium": "7440-70-2",
    "Cadmium": "7440-43-9",
    "Iron": "7439-89-6",
    "Chromium": "7440-47-3",
    "Copper": "7440-50-8",
    "Magnesium": "7439-95-4",
    "Phosphorus": "7723-14-0",
    "Potassium": "7440-09-7",
    "Manganese": "7439-96-5",
    "Sodium": "7440-23-5",
    "Chloromethane": "74-87-3",
    "Vinyl chloride": "75-01-4",
    "Bromomethane": "74-83-9",
    "Chloroethane": "75-00-3",
    "Trichlorofluoromethane": "75-69-4",
    "1,1-Dichloroethene": "75-35-4",
    "Acetone": "67-64-1",
    "Iodomethane": "74-88-4",
    "Carbon disulfide": "75-15-0",
    "Methylene chloride": "75-09-2",
    "Acrylonitrile": "107-13-1",
    "1,1-Dichloroethane": "75-34-3",
    "Vinyl acetate": "108-05-4",
    "2-Butanone": "78-93-3",
    "cis-1,2-Dichloroethene": "156-59-2",
    "Bromochloromethane": "74-97-5",
    "Chloroform": "67-66-3",
    "1,1,1-Trichloroethane": "71-55-6",
    "Carbon tetrachloride": "56-23-5",
    "Benzene": "71-43-2",
    "1,2-Dichloroethane": "107-06-2",
    "Trichloroethene": "79-01-6",
    "1,2-Dichloropropane": "78-87-5",
    "Dibromomethane": "74-95-3",
    "Bromodichloromethane": "75-27-4",
    "cis-1,3-Dichloropropene": "10061-01-5",
    "4-Methyl-2-pentanone": "108-10-1",
    "Toluene": "108-88-3",
    "trans-1,3-Dichloropropene": "10061-02-6",
    "1,1,2-Trichloroethane": "79-00-5",
    "Tetrachloroethene": "127-18-4",
    "2-Hexanone": "591-78-6",
    "Dibromochloromethane": "124-48-1",
    "1,2-Dibromoethane (EDB)": "106-93-4",
    "Chlorobenzene": "108-90-7",
    "1,1,1,2-Tetrachloroethane": "630-20-6",
    "Ethylbenzene": "100-41-4",
    "m,p-Xylene": "179601-23-1",
    "o-Xylene": "95-47-6",
    "Xylenes, total": "1330-20-7",
    "Styrene": "100-42-5",
    "Bromoform": "75-25-2",
    "1,1,2,2-Tetrachloroethane": "79-34-5",
    "1,2,3-Trichloropropane": "96-18-4",
    "trans-1,4-Dichloro-2-butene": "110-57-6",
    "1,4-Dichlorobenzene": "106-46-7",
    "1,2-Dichlorobenzene": "95-50-1",
    "1,2-Dibromo-3-chloropropane": "96-12-8",
    "1,2-Dichloroethane-d4": "17060-07-0",
    "Toluene-d8": "2037-26-5",
    "4-Bromofluorobenzene": "460-00-4",
    "1,2-Dichlorobenzene-d4": "2199-69-1",
    "Total Inorganic Nitrogen": "InorgN",
    "Ammonia as N": "7664-41-7",
    "Total Dissolved Solids": "TDS",
    "Alkalinity, bicarbonate, to pH 4.5": "ALKB4.5",
    "Total Organic Carbon": "TOC",
    "Chloride": "16887-00-6",
    "Nitrate as N": "14797-55-8",
    "Nitrite as N": "14797-65-0",
    "Sulfate as SO4": "14808-79-8",
    "Barium": "7440-39-3",
    "Antimony": "7440-36-0",
    "Lithium": "7439-93-2",
    "Chemical Oxygen Demand": "COD",
    "Selenium": "7782-49-2"
}

In [30]:
def assign_matrix(sample_id):
    sample_str = str(sample_id).strip()
    
    # 1. Groundwater wells
    if sample_str.startswith('MW-'):
        return 'WG'
    
    # 2. Specific site locations
    elif sample_str == 'Leachate':
        return 'LE'
    elif sample_str == 'UNDERDRAIN':
        return 'WS'
    elif sample_str in ['GROESBECK POND', 'COOPER-1', 'COOPER-2']:
        return 'WS'
    
    # 3. QA/QC Samples
    elif 'Duplicate' in sample_str:
        return 'WG'  # Field duplicates inherit parent matrix
    elif 'Blank' in sample_str:
        return 'WQ'  # Trip, Equipment, and Field blanks stay WQ
    
    else:
        return 'U' # Unknown/Unassigned

In [31]:
lab_matrix_dict = {
    'Ground Water': 'WG',
    'Aqueous': 'WU'
}

In [32]:
edd_cols_to_template_dict = {
    "LABID": "lab_sample_id",
    "SAMPLEID": "sys_loc_code",  # mapped to sample_name, but cross-check with #sys_sample_code
    "DATESAMPLED": "sample_date",
    "DATERECEIVED": "sample_comments",  # Standard templates often drop 'received date' into comments if there isn't a dedicated column
    "DATEEXTRACTED": "prep_date",
    "DATEANALYZED": "analysis_date",
    "MATRIX": "lab_matrix",  # check if it fits 'lab_matrix' or 'sample_matrix_code' better in your system
    "METHOD": "analytic_method",
    "PARAMETER": "chemical_name",  # cross-check if your system prefers 'cas_rn' or 'chemical_name'
    "RESULTS": "result_value",
    "UNITS": "result_unit",
    "DETECTIONLIMITS": "reporting_detection_limit",  # could also be 'method_detection_limit' depending on the lab
    "NOTES": "result_comments",
    "DILUTION": "dilution_factor",
    "Total Or Dissolved": "fraction",  # Total/Dissolved maps to the 'fraction' column
    "MCL": "ProjectSpecificQualifiers",  # Maximum Contaminant Level (MCL) doesn't have a direct slot, usually goes to custom/project fields
}

In [33]:
sample_id_dict = {
    "Under Drain": "UNDERDRAIN",
    "Leachate": "LEACHATE",
    'Groesbeck Pond': 'GROESBECK POND',
    'Cooper 1': 'COOPER-1',
    'Cooper 2': 'COOPER-2',
    'MW-3sr': 'MW-03sr',
    'MW-3dr': 'MW-03dr',
    'MW-5sr': 'MW-05sr',
    'MW-5dr': 'MW-05dr',
    'MW-6dr': 'MW-06dr',
    'MW-6sr': 'MW-06sr',

}

In [34]:
import pandas as pd
import numpy as np

EDD = _26D0675.copy()
EDD.rename(columns=edd_cols_to_template_dict, inplace=True)

# 1. Handle Datetimes safely
temp_datetime = pd.to_datetime(_26D0675["DATESAMPLED"], errors='coerce')
EDD["sample_date"] = temp_datetime.dt.strftime("%m/%d/%Y")  # 04/13/2026
EDD["sample_time"] = temp_datetime.dt.strftime("%H:%M:%S")  # 09:10:00

# 2. Preserve sample_name BEFORE modifying sys_loc_code, then assign matrix
EDD['sample_name'] = EDD['sys_loc_code']

# mappings
EDD['analytic_method'] = _26D0675['METHOD'].map(edd_to_db_methods)
EDD["chemical_name"] = _26D0675['PARAMETER'].map(chemical_name_dict).fillna(_26D0675['PARAMETER'])
EDD['cas_rn'] = EDD['chemical_name'].map(analytes_cas_dict).fillna("Unknown TIC")
EDD['sys_loc_code'] = _26D0675['SAMPLEID'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  # Map to standardized sample names if possible, else keep original code

EDD['sample_matrix_code'] = EDD['sample_name'].apply(assign_matrix)
EDD['lab_matrix'] = _26D0675['MATRIX'].map(lab_matrix_dict).fillna(EDD['sample_matrix_code'])  # Map to lab_matrix if possible, else keep original code

# 3. Clean QC samples out of sys_loc_code without altering text casing globally
######################################################################################
blank_these_locs = ["equipment blank", "field blank a", "field blank b", "trip blank", "field blank"]
######################################################################################
is_qc_blank = EDD["sys_loc_code"].astype(str).str.strip().str.lower().isin(blank_these_locs)
EDD.loc[is_qc_blank, "sys_loc_code"] = ""

# 4. Generate #sys_sample_code safely
date_yyyymmdd = pd.to_datetime(EDD["sample_date"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
EDD['#sys_sample_code'] = EDD['sample_name'].str.strip() + "_" + date_yyyymmdd

# 5. Define Duplicate Hierarchies
duplicate_01_ID, parent_01_sample_id = "Duplicate A", "MW-33s"
duplicate_02_ID, parent_02_sample_id = "Duplicate B", "MW-30s"
duplicate_03_ID, parent_03_sample_id = "Duplicate C", "MW-41d"

# Create clean conditional masks
cond_01 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_01_ID.upper()
cond_02 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_02_ID.upper()
cond_03 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_03_ID.upper()

cond_01b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_01_sample_id.upper()
cond_02b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_02_sample_id.upper()
cond_03b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_03_sample_id.upper()

# Map Parent Sample Codes

EDD.loc[cond_01, 'parent_sample_code'] = EDD.loc[cond_01b, "#sys_sample_code"][EDD.loc[cond_01b, "#sys_sample_code"].first_valid_index()]
EDD.loc[cond_02, 'parent_sample_code'] = EDD.loc[cond_02b, '#sys_sample_code'][EDD.loc[cond_02b, '#sys_sample_code'].first_valid_index()]
EDD.loc[cond_03, 'parent_sample_code'] = EDD.loc[cond_03b, '#sys_sample_code'][EDD.loc[cond_03b, '#sys_sample_code'].first_valid_index()]

# FIX: Use the existing robust masks to update sys_loc_code safely
EDD.loc[cond_01, "sys_loc_code"] = parent_01_sample_id
EDD.loc[cond_02, "sys_loc_code"] = parent_02_sample_id
EDD.loc[cond_03, "sys_loc_code"] = parent_03_sample_id

# 6. FIX: Use np.select to evaluate sample types conditionally without overwriting
sample_name_lower = EDD["sample_name"].astype(str).str.lower()
conditions = [
    sample_name_lower.str.contains("equipment"),
    sample_name_lower.str.contains("trip"),
    sample_name_lower.str.contains("duplicate"),
    sample_name_lower.str.contains("field")
]
choices = ["EB", "TB", "FD", "FB"]
EDD['sample_type_code'] = np.select(conditions, choices, default="N")

# 7. Data normalization
EDD['detect_flag'] = EDD['result_value'].apply(lambda x: "N" if pd.isnull(x) or x == "ND" or x == pd.NaT else "Y")
EDD['result_value'] = EDD['result_value'].apply(lambda x: None if x in ["ND", pd.NaT] or pd.isnull(x) else x)
EDD['result_unit'] = EDD['result_value'].apply(lambda x: None if pd.isnull(x) else "ng/L") # Note: Ensure your DB allows null units for NDs

# Final Fill-Ins for Required Fields
EDD['result_type_code'] = "TRG"
EDD['reportable_result'] = "Y"
EDD['test_type'] = "Initial"
EDD['Lab_SDG'] = lab_sgd
EDD['prep_method'] = "Method"

# Fill in blank fraction with "N"
EDD['fraction'] = EDD['fraction'].fillna("N")

# In one line, fix the dilution factor column to not exceed 1 decimal place, but also not convert integers to floats unnecessarily
EDD['dilution_factor'] = EDD['dilution_factor'].round(1).where(EDD['dilution_factor'].notnull(), None)


# if result value is "<" or "" set detect_flag to "N" and result_value to ""
EDD.loc[EDD['result_value'] == "<", 'result_value'] = None
EDD.loc[EDD['result_value'] == None, 'result_unit'] = None
EDD.loc[EDD['result_value'].isnull(), 'detect_flag'] = "N"

# 8. FIX: Reindex at the very end to ensure temporary calculation spaces aren't mutated prematurely
EDD = EDD.reindex(columns=template_df.columns)

In [36]:
# 1. Create a lookup dictionary of the correct units for each analyte.
# This filters out the blanks and finds the first valid unit for every chemical name.
analyte_unit_lookup = (
    EDD.dropna(subset=['result_unit'])
    .groupby('chemical_name')['result_unit']
    .first()
    .to_dict()
)

# 2. Use np.where to dynamically assign the unit:
# - If 'reporting_detection_limit' is missing -> set to None
# - Otherwise -> look up the correct unit based on the 'chemical_name'
EDD['detection_limit_unit'] = np.where(
    EDD['reporting_detection_limit'].isna(),
    None,
    EDD['chemical_name'].map(analyte_unit_lookup)
)

# Map units by analyte, then null where reporting_detection_limit is missing
EDD['detection_limit_unit'] = EDD['chemical_name'].map(analyte_unit_lookup)
EDD.loc[EDD['reporting_detection_limit'].isna(), 'detection_limit_unit'] = None
EDD[EDD.sample_name.str.contains('Duplicate')][['#sys_sample_code','sys_loc_code', 'chemical_name', 'reporting_detection_limit', 'detection_limit_unit', 'parent_sample_code']]

,#sys_sample_code,sys_loc_code,chemical_name,reporting_detection_limit,detection_limit_unit,parent_sample_code
589,Duplicate B_20260416,MW-30s,Boron,20.0,ng/L,MW-30s_20260414
590,Duplicate B_20260416,MW-30s,Cadmium,0.2,ng/L,MW-30s_20260414
591,Duplicate B_20260416,MW-30s,Potassium,500.0,ng/L,MW-30s_20260414
592,Duplicate B_20260416,MW-30s,Sodium,1000.0,ng/L,MW-30s_20260414
593,Duplicate B_20260416,MW-30s,Chloride,10.0,ng/L,MW-30s_20260414
594,Duplicate B_20260416,MW-30s,Ammonia as N,10.0,ng/L,MW-30s_20260414
595,Duplicate B_20260416,MW-30s,Total Organic Carbon,700.0,ng/L,MW-30s_20260414
678,Duplicate C_20260420,MW-41d,Boron,20.0,ng/L,MW-41d_20260420
679,Duplicate C_20260420,MW-41d,Cadmium,0.2,ng/L,MW-41d_20260420
680,Duplicate C_20260420,MW-41d,Potassium,500.0,ng/L,MW-41d_20260420


In [37]:
# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{file_name.split('.')[0]}_ESBasic_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="ESBasic_TRC",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

In [38]:
## Mapping script to find the best matches between your EDD methods and the database codes, and print a dictionary for you to copy/paste into your main script. This is a helper script to make the mapping process easier, especially when method names don't match exactly but are close enough to be recognizable.
import pandas as pd
import re

# 1. Your new EDD methods
edd_methods = [
    'EPA 200.8 Rev. 5.4', 'EPA 200.7 Rev. 4.4', 'EPA 8260D', 'Calculation', 
    'EPA 350.1 Rev. 2.0', 'SM 2540 C-20', 'SM 2320 B-21', 'SM 5310 B-14', 
    'EPA 300.0 Rev. 2.1', 'SM 4500-Cl D-21', 'EPA 410.4 Rev. 2.0'
]

# 2. Load your reference file and grab the first column
try:
    df_ref = pd.read_excel('rt_analytic_method.xlsx')
    db_codes = df_ref.iloc[:, 0].dropna().astype(str).unique().tolist()
except FileNotFoundError:
    print("Error: Could not find the file. Make sure the script and CSV are in the same folder.")
    db_codes = []

# 3. Helper function to strip out spaces, punctuation, and "Rev" versions for matching
def normalize(text):
    text = str(text).upper().replace(" ", "")
    text = re.sub(r'REV\..*$', '', text) # Strip "Rev. X.X"
    text = re.sub(r'[^A-Z0-9]', '', text) # Keep only alphanumeric
    return text

method_mapping = {}

# 4. Find the best match
if db_codes:
    for edd_m in edd_methods:
        norm_edd = normalize(edd_m)
        matched = False
        
        for db_c in db_codes:
            norm_db = normalize(db_c)
            # Check if the core letters/numbers match
            if norm_edd in norm_db or norm_db in norm_edd:
                method_mapping[edd_m] = db_c
                matched = True
                break
                
        # Flag any methods that didn't find a clean match
        if not matched:
            method_mapping[edd_m] = "MANUAL_CHECK_REQUIRED"

    # 5. Print the beautifully formatted dictionary
    print("Here is your dictionary:\n")
    print("edd_to_db_methods = {")
    for key, value in method_mapping.items():
        print(f"    '{key}': '{value}',")
    print("}")

Here is your dictionary:

edd_to_db_methods = {
    'EPA 200.8 Rev. 5.4': 'MANUAL_CHECK_REQUIRED',
    'EPA 200.7 Rev. 4.4': 'MANUAL_CHECK_REQUIRED',
    'EPA 8260D': 'MANUAL_CHECK_REQUIRED',
    'Calculation': 'CALC',
    'EPA 350.1 Rev. 2.0': 'MANUAL_CHECK_REQUIRED',
    'SM 2540 C-20': 'SM-2540',
    'SM 2320 B-21': 'SM2320',
    'SM 5310 B-14': 'SM5310',
    'EPA 300.0 Rev. 2.1': 'MANUAL_CHECK_REQUIRED',
    'SM 4500-Cl D-21': 'SM4500',
    'EPA 410.4 Rev. 2.0': 'MANUAL_CHECK_REQUIRED',
}
